6241947b-fc3e-436a-b66e-2e432ff8e480

# CMA-ES Optimization for Worst-Case OMWU Dynamics
This notebook demonstrates how to use the `CMAESGameOptimizer` to find payoff matrices that cause the Optimistic Multiplicative Weights Update (OMWU) algorithm to suffer maximum asymptotic regret in a 2-player, 2-action game.


In [1]:
import sys
sys.path.append("..")

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.config.schemas import ExperimentConfig, GameConfig, DynamicConfig, ExecutionConfig, CMAESConfig
from src.engine.optimizer import CMAESGameOptimizer
from src.engine.runner import ExperimentRunner
from src.engine.statistics import load_experiment_stats

# Set formatting for plots
plt.style.use('ggplot')


## 1. Configure the Optimizer
We configure a 2x2 game base template and set the CMA-ES optimizer to target the `delta_reg` objective. This helps us find games where the algorithm continues to diverge or cycle heavily even late into the simulation.


In [2]:
# Setup a small configuration for 2x2 games
# We use a batch size of 20 to run 20 games in parallel per generation

config = ExperimentConfig(
    name="cmaes_omwu_demo",
    game=GameConfig(
        generator="random",
        action_sizes = [2, 2],
        seed=42,
        utility_range=(-1.0, 1.0),
        # payoffs=[
        #     [[0.0, 0.0], [0.0, 0.0]], # Player 1 base
        #     [[0.0, 0.0], [0.0, 0.0]], # Player 2 base
        # ]
    ),
    dynamic=DynamicConfig(
        algorithm="omwu", 
        eta=0.05,
        logit_penalty_threshold=10,
        logit_penalty_norm=2,
        logit_penalty_mode="centered"

    ),
    execution=ExecutionConfig(
        total_steps=40000,
        steps_per_call=200,
        device="auto",
        dtype="float64", # Run CMA-ES in double precision to avoid numerical artifacts!
        compile=True
    ),
    cmaes=CMAESConfig(
        sigma=0.5,
        seed=42,
        objective_type="envelope_trend_log",
        # objective_type="raw",
        population_size=40,             # Number of games evaluated in parallel per generation
        maxiter=5,                      # Max generations per single IPOP restart
        maxfevals=50000,                # Total budget of function evaluations across all restarts
        tolfun=1e-4,                    # Stop generation early if flat-fitness tolerance is reached
        restarts=1,                     # Number of allowed IPOP (Increasing Population) restarts
        T1_ratio=0.8,
        lambda_reg=1,
        gamma_volatility=1.5,
        logit_penalty_weight=20,
        logit_penalty_average=True
    )
)

# Initialize CMA-ES Optimizer
# T1_ratio = 0.5 means we compute delta regret from step 500 to 1000
optimizer = CMAESGameOptimizer(
    base_config=config
)


## 2. Run the Optimization
We run the optimizer for a few generations (e.g., 5). In a real experiment, you might run this for 50-100 generations.


In [3]:
# Run optimization
best_payoffs, best_factors = optimizer.optimize()

print("Optimization Complete!")
import json
print("\nCMA-ES Final Objective Breakdown:")
print(json.dumps({k: float(v) for k, v in best_factors.items()}, indent=2))
print("\nWorst-case Payoff Matrix Player 1:")
print(np.round(best_payoffs[0].numpy(), 4))
print("\nWorst-case Payoff Matrix Player 2:")
print(np.round(best_payoffs[1].numpy(), 4))
print(optimizer.session_id)


[09/04/26 11:00:52] INFO     Starting simulation 'cmaes_omwu_demo' [Session: 927e4e90] on device 'cuda' for T=40000
                             steps.

KeyboardInterrupt: 

## 2.5 Reloading Past Optimization Results
Because CMA-ES optimizations can take a long time, the `CMAESGameOptimizer` automatically saves the best matrices and config at the end of `optimize()` to `outputs/cmaes_best_{session_id}.pt`. 

You can easily reload these exact tensors later using the static `load_results` method!

In [ ]:
# The session_id was generated during initialization
session_id = optimizer.session_id

# We can load the exact dict saved to disk
saved_data = CMAESGameOptimizer.load_results(session_id)

# Verify it matches perfectly
assert torch.allclose(best_payoffs[0], saved_data["payoffs"][0])
print(f"Successfully reloaded results for session {session_id}!")

# To load from a completely past run, you would just type:
# past_data = CMAESGameOptimizer.load_results("1a2b3c4d")
# best_payoffs = past_data["payoffs"]

## 3. Simulate the Worst-Case Game
Now we plug these worst-case matrices back into a standard `ExperimentRunner` and run a longer, high-fidelity simulation (e.g. 5000 steps) to see what the dynamics look like.


In [ ]:
best_payoffs



In [ ]:
import copy

# Create a validation config with the worst-case matrices
val_config = copy.deepcopy(config)
val_config.parent_session_id = session_id
val_config.dynamic.eta = 0.05
val_config.game.payoffs = [p.numpy().tolist() for p in best_payoffs]
val_config.execution.batch_size = 1
val_config.execution.total_steps = 400000
val_config.execution.steps_per_call = 200
val_config.game.generator = 'custom'
val_config.name = "worst_case_omwu"

# Run the single simulation
runner = ExperimentRunner(val_config)
summary = runner.run()

print(f"Validation Run Complete. Session ID: {summary['session_id']}")


## 4. Plot the Trajectories
We extract the cumulative regret and strategy probabilities using `load_experiment_stats` and visualize the non-converging dynamics.


In [ ]:
from src.engine.statistics import unpack_stats
from src.utils.visualization import plot_static_trajectories

# Load and unpack the recorded statistics from disk
stats_data = load_experiment_stats(output_dir='outputs', session_id=summary['session_id'])
steps, cum_regrets, strats, logits, instant_payoffs = unpack_stats(stats_data)

# Plot the static N-player trajectories (Zoom in by setting start_step and end_step if desired!)
import torch
cum_action_payoffs = [torch.cumsum(p, dim=0) for p in instant_payoffs]
cum_expected_payoffs = [torch.cumsum(torch.sum(s * p, dim=-1), dim=0) for s, p in zip(strats, instant_payoffs)]
fig, axes = plot_static_trajectories(steps, cum_regrets, strats, title_prefix="OMWU", plot_all_actions=True, start_step=15000, end_step=None, cum_action_payoffs=cum_action_payoffs, cum_expected_payoffs=cum_expected_payoffs, logits=logits)
plt.show()
